# Official PARSeq ANPR Fine-Tuning With Iterative Refinement

Notebook nay tu chua toan bo phan kien truc va helper can thiet de fine-tune official PARSeq cho ANPR. Khong import tu file/module noi bo cua repo; notebook chi import cac thu vien Python/PyTorch ben ngoai.

Muc tieu:

- Load official PARSeq pretrained weight (`parseq-bb5792a6.pt`).
- Giu co che `decode_ar` va `refine_iters` cua PARSeq goc.
- Fine-tune tren `ocr_dataset_rescued_bbox_new` bang PLM loss tuong duong `PARSeq.training_step`.
- Danh gia ablation iterative refinement: `refine_iters = 0, 1, 2, 3` truoc va sau fine-tune.


In [9]:
from __future__ import annotations

import json
import math
import random
import time
from dataclasses import asdict, dataclass
from functools import partial
from itertools import permutations
from pathlib import Path
from types import SimpleNamespace
from typing import Iterable, Optional, Sequence

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image, ImageEnhance, ImageOps
from torch import Tensor, nn
from torch.nn.modules import transformer
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms as T
from tqdm.auto import tqdm

def _to_2tuple(value):
    if isinstance(value, (tuple, list)):
        return tuple(value)
    return (value, value)


class PatchEmbed(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_chans=3, embed_dim=768):
        super().__init__()
        img_size = _to_2tuple(img_size)
        patch_size = _to_2tuple(patch_size)
        self.grid_size = (img_size[0] // patch_size[0], img_size[1] // patch_size[1])
        self.num_patches = self.grid_size[0] * self.grid_size[1]
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        return self.proj(x).flatten(2).transpose(1, 2)


class Mlp(nn.Module):
    def __init__(self, in_features, hidden_features=None, out_features=None, act_layer=nn.GELU, drop=0.0):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = act_layer()
        self.drop1 = nn.Dropout(drop)
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop2 = nn.Dropout(drop)

    def forward(self, x):
        return self.drop2(self.fc2(self.drop1(self.act(self.fc1(x)))))


class Attention(nn.Module):
    def __init__(self, dim, num_heads=8, qkv_bias=False, attn_drop=0.0, proj_drop=0.0):
        super().__init__()
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim**-0.5
        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

    def forward(self, x):
        batch_size, seq_len, channels = x.shape
        qkv = self.qkv(x).reshape(batch_size, seq_len, 3, self.num_heads, channels // self.num_heads)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = self.attn_drop(attn.softmax(dim=-1))
        x = (attn @ v).transpose(1, 2).reshape(batch_size, seq_len, channels)
        return self.proj_drop(self.proj(x))


class TransformerBlock(nn.Module):
    def __init__(self, dim, num_heads, mlp_ratio=4.0, qkv_bias=True, drop=0.0, attn_drop=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = Attention(dim, num_heads=num_heads, qkv_bias=qkv_bias, attn_drop=attn_drop, proj_drop=drop)
        self.drop_path = nn.Identity()
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = Mlp(in_features=dim, hidden_features=int(dim * mlp_ratio), drop=drop)

    def forward(self, x):
        x = x + self.drop_path(self.attn(self.norm1(x)))
        x = x + self.drop_path(self.mlp(self.norm2(x)))
        return x


class VisionTransformer(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_chans=3, embed_dim=768, depth=12, num_heads=12, mlp_ratio=4.0, qkv_bias=True, drop_rate=0.0, attn_drop_rate=0.0, drop_path_rate=0.0, embed_layer=PatchEmbed, num_classes=0, global_pool="", class_token=False):
        super().__init__()
        if class_token:
            raise ValueError("The embedded PARSeq encoder expects class_token=False.")
        self.patch_embed = embed_layer(img_size=img_size, patch_size=patch_size, in_chans=in_chans, embed_dim=embed_dim)
        self.pos_embed = nn.Parameter(torch.zeros(1, self.patch_embed.num_patches, embed_dim))
        self.pos_drop = nn.Dropout(p=drop_rate)
        self.blocks = nn.Sequential(*[TransformerBlock(embed_dim, num_heads, mlp_ratio, qkv_bias, drop_rate, attn_drop_rate) for _ in range(depth)])
        self.norm = nn.LayerNorm(embed_dim)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.trunc_normal_(module.weight, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.LayerNorm):
            nn.init.zeros_(module.bias)
            nn.init.ones_(module.weight)

    def no_weight_decay(self):
        return {"pos_embed"}

    def forward_features(self, x):
        x = self.patch_embed(x)
        x = self.pos_drop(x + self.pos_embed)
        x = self.blocks(x)
        return self.norm(x)

ANPR_CHARSET = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ"
PARSEQ_94_FULL_CHARSET = "0123456789abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ!\"#$%&'()*+,-./:;<=>?@[\\]^_`{|}~"
PRETRAINED_PARSEQ_URL = "https://github.com/baudm/parseq/releases/download/v1.0.0/parseq-bb5792a6.pt"
def find_repo_dir(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "ocr_dataset_rescued_bbox_new").exists() or (candidate / "baseline_icpr_2026").exists():
            return candidate
    return start

REPO_DIR = find_repo_dir(Path.cwd())
WORKSPACE_DIR = REPO_DIR.parent if REPO_DIR.name.lower() == "lpr" else REPO_DIR
PIPELINE_DIR = REPO_DIR / "parseq_official_pipeline"
DEFAULT_DATA_ROOT = REPO_DIR / "ocr_dataset_rescued_bbox_new"
DEFAULT_OUTPUT_DIR = PIPELINE_DIR / "outputs" / "refinement_finetune"
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("repo", REPO_DIR)
print("device", DEVICE)


@dataclass
class OfficialPARSeqANPRConfig:
    data_root: str = str(DEFAULT_DATA_ROOT)
    dataset_sources: Optional[list[dict]] = None
    output_dir: str = str(DEFAULT_OUTPUT_DIR)
    experiment: str = "parseq"
    pretrained: bool = True
    decode_ar: bool = True
    refine_iters: int = 1
    charset_test: str = ANPR_CHARSET
    img_size: tuple[int, int] = (32, 128)
    max_label_length: int = 12
    preprocess: bool = False
    augment: bool = True
    batch_size: int = 16
    num_workers: int = 0
    epochs: int = 5
    lr: float = 1e-5
    weight_decay: float = 1e-4
    grad_clip: float = 20.0
    seed: int = 42
    amp: bool = True
    split_val_ratio: float = 0.10
    split_test_ratio: float = 0.10
    exclude_rejected_labels: bool = True
    balance_plate_types: bool = True


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def normalize_plate_text(text: object) -> str:
    return "".join(ch for ch in str(text).upper() if ch in ANPR_CHARSET)


def _as_repo_path(path_like: object) -> Path:
    path = Path(str(path_like))
    return path if path.is_absolute() else (REPO_DIR / path)


def _resolve_relative_path(root: Path, path_like: object) -> Path:
    path = Path(str(path_like))
    return path if path.is_absolute() else (root / path)


def _is_nonempty(value: object) -> bool:
    if value is None or pd.isna(value):
        return False
    return bool(str(value).strip())


def _is_rejected(row: pd.Series) -> bool:
    statuses = [str(row.get(col, "")).strip().lower() for col in ("review_status", "label_status")]
    return any(status == "rejected" for status in statuses)


def _pick_reviewed_ocr_label(row: pd.Series, exclude_rejected: bool = True) -> str:
    if exclude_rejected and _is_rejected(row):
        return ""
    for col in ("corrected_label", "extracted_character", "prediction", "raw_prediction", "original_label", "text"):
        if col in row and _is_nonempty(row[col]):
            label = normalize_plate_text(row[col])
            if label:
                return label
    return ""


def _filter_and_finalize_frame(frame: pd.DataFrame, max_label_length: int, source_name: str) -> pd.DataFrame:
    if frame.empty:
        return frame
    frame = frame.copy()
    frame["label"] = frame["label"].map(normalize_plate_text)
    frame = frame[frame["label"].str.len().between(1, int(max_label_length))].copy()
    frame["image_path"] = frame["image_path"].map(lambda value: str(Path(value)))
    exists = frame["image_path"].map(lambda value: Path(value).exists())
    if not bool(exists.all()):
        print(f"warning: skipped {int((~exists).sum())} missing images from {source_name}")
        frame = frame[exists].copy()
    return frame.reset_index(drop=True)


def _split_unsplit_frame(frame: pd.DataFrame, seed: int, val_ratio: float, test_ratio: float, stratify_col: str = "plate_type") -> pd.DataFrame:
    frame = frame.copy().reset_index(drop=True)
    if frame.empty:
        frame["split"] = []
        return frame
    rng = np.random.default_rng(int(seed))
    splits = np.full(len(frame), "train", dtype=object)
    if stratify_col in frame.columns:
        groups = frame.groupby(stratify_col, sort=True).indices.values()
    else:
        groups = [np.arange(len(frame))]
    for positions in groups:
        idx = np.asarray(list(positions), dtype=int)
        rng.shuffle(idx)
        n = len(idx)
        n_test = max(1, int(round(n * test_ratio))) if n >= 3 and test_ratio > 0 else 0
        n_val = max(1, int(round(n * val_ratio))) if n >= 3 and val_ratio > 0 else 0
        if n_test + n_val >= n:
            n_test = 1 if n >= 3 and test_ratio > 0 else 0
            n_val = 1 if n - n_test >= 2 and val_ratio > 0 else 0
        splits[idx[:n_test]] = "test"
        splits[idx[n_test:n_test + n_val]] = "val"
    frame["split"] = splits
    return frame


def _read_prepared_csv_source(source: dict, cfg: OfficialPARSeqANPRConfig) -> pd.DataFrame:
    root = _as_repo_path(source["path"])
    source_name = source.get("name", root.name)
    plate_type = source.get("plate_type", source_name)
    frames = []
    for split in ("train", "val", "test"):
        csv_path = root / f"{split}.csv"
        if not csv_path.exists():
            csv_path = root / "splits" / f"{split}.csv"
        if not csv_path.exists():
            continue
        frame = pd.read_csv(csv_path)
        image_col = "image_path" if "image_path" in frame.columns else "image"
        if image_col not in frame.columns or "label" not in frame.columns:
            raise ValueError(f"{csv_path} needs image/image_path and label columns")
        labels = frame["label"].astype(str)
        if labels.str.lower().str.endswith(".txt").mean() > 0.5:
            raise ValueError(f"{csv_path} label column looks like label-file paths, not OCR text")
        out = pd.DataFrame({
            "split": split,
            "image_path": frame[image_col].map(lambda value: str(_resolve_relative_path(root, value))),
            "label": frame["label"],
            "source_name": source_name,
            "plate_type": plate_type,
            "label_status": frame.get("label_status", "prepared"),
            "review_status": frame.get("review_status", "prepared"),
        })
        frames.append(out)
    if not frames:
        raise FileNotFoundError(f"No split CSV files found in {root}")
    return _filter_and_finalize_frame(pd.concat(frames, ignore_index=True), cfg.max_label_length, source_name)


def _read_collected_review_source(source: dict, cfg: OfficialPARSeqANPRConfig) -> pd.DataFrame:
    root = _as_repo_path(source["path"])
    source_name = source.get("name", root.name)
    plate_type = source.get("plate_type", source_name)
    csv_path = root / "labels.csv"
    if not csv_path.exists():
        raise FileNotFoundError(csv_path)
    frame = pd.read_csv(csv_path)
    image_col = "image" if "image" in frame.columns else "image_path"
    if image_col not in frame.columns:
        raise ValueError(f"{csv_path} needs an image or image_path column")
    out = pd.DataFrame({
        "split": frame.get("split", "train"),
        "image_path": frame[image_col].map(lambda value: str(_resolve_relative_path(root, value))),
        "label": frame.apply(lambda row: _pick_reviewed_ocr_label(row, cfg.exclude_rejected_labels), axis=1),
        "source_name": source_name,
        "plate_type": plate_type,
        "label_status": frame.get("label_status", "reviewed"),
        "review_status": frame.get("review_status", "reviewed"),
    })
    out = out[out["label"].astype(str).str.len() > 0].copy()
    return _filter_and_finalize_frame(out, cfg.max_label_length, source_name)


def _iter_labels_txt_dirs(root: Path):
    if (root / "labels.txt").exists():
        yield root
    for child in sorted(root.iterdir() if root.exists() else []):
        if child.is_dir() and (child / "labels.txt").exists():
            yield child


def _read_labels_txt_source(source: dict, cfg: OfficialPARSeqANPRConfig) -> pd.DataFrame:
    root = _as_repo_path(source["path"])
    source_name = source.get("name", root.name)
    rows = []
    for labels_dir in _iter_labels_txt_dirs(root):
        plate_type = source.get("plate_type") if labels_dir == root else labels_dir.name
        plate_type = plate_type or labels_dir.name
        for line in (labels_dir / "labels.txt").read_text(encoding="utf-8").splitlines():
            if not line.strip():
                continue
            parts = line.strip().replace("	", " ").split()
            if len(parts) < 2:
                continue
            image_path = labels_dir / parts[0]
            rows.append({
                "image_path": str(image_path),
                "label": parts[1],
                "source_name": source_name,
                "plate_type": plate_type,
                "label_status": "labels_txt",
                "review_status": "labels_txt",
            })
    if not rows:
        raise FileNotFoundError(f"No labels.txt files found in {root}")
    frame = _split_unsplit_frame(pd.DataFrame(rows), cfg.seed, cfg.split_val_ratio, cfg.split_test_ratio)
    return _filter_and_finalize_frame(frame, cfg.max_label_length, source_name)


def _detect_source_format(root: Path) -> str:
    if (root / "labels.csv").exists():
        return "collected_review_csv"
    if (root / "train.csv").exists() or (root / "splits" / "train.csv").exists():
        return "prepared_csv"
    if any(True for _ in _iter_labels_txt_dirs(root)):
        return "labels_txt"
    raise ValueError(f"Could not detect dataset format for {root}")


def build_plate_dataframe(cfg: OfficialPARSeqANPRConfig) -> pd.DataFrame:
    sources = cfg.dataset_sources or [{"name": "data_root", "path": cfg.data_root, "format": "prepared_csv", "plate_type": "normal"}]
    frames = []
    readers = {
        "prepared_csv": _read_prepared_csv_source,
        "collected_review_csv": _read_collected_review_source,
        "labels_txt": _read_labels_txt_source,
        "color_filtered_labels_txt": _read_labels_txt_source,
    }
    for source in sources:
        root = _as_repo_path(source["path"])
        if not root.exists():
            print(f"warning: skipped missing dataset source {root}")
            continue
        source = dict(source)
        fmt = source.get("format") or _detect_source_format(root)
        if fmt not in readers:
            raise ValueError(f"Unsupported dataset format: {fmt}")
        frame = readers[fmt](source, cfg)
        frames.append(frame)
        print(f"loaded {len(frame):5d} rows from {source.get('name', root.name)} ({fmt})")
    if not frames:
        raise ValueError("No usable dataset rows were loaded")
    frame = pd.concat(frames, ignore_index=True)
    frame = frame.drop_duplicates(subset=["image_path", "label"]).reset_index(drop=True)
    return frame


def summarize_dataset_frame(frame: pd.DataFrame) -> pd.DataFrame:
    if frame.empty:
        return pd.DataFrame(columns=["split", "plate_type", "source_name", "samples"])
    return (
        frame.groupby(["split", "plate_type", "source_name"], dropna=False)
        .size()
        .reset_index(name="samples")
        .sort_values(["split", "plate_type", "source_name"])
        .reset_index(drop=True)
    )


def preprocess_plate_image(image: Image.Image) -> Image.Image:
    try:
        import cv2
    except Exception:
        gray = ImageOps.grayscale(image)
        gray = ImageOps.autocontrast(gray)
        gray = ImageEnhance.Sharpness(gray).enhance(1.6)
        return Image.merge("RGB", (gray, gray, gray))

    rgb = image.convert("RGB")
    arr = np.asarray(rgb)
    gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    gray = clahe.apply(gray)
    gray = cv2.bilateralFilter(gray, 5, 50, 50)
    blur = cv2.GaussianBlur(gray, (0, 0), sigmaX=1.0)
    sharp = cv2.addWeighted(gray, 1.5, blur, -0.5, 0)
    return Image.fromarray(np.stack([sharp, sharp, sharp], axis=-1))


class PlateCropDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, split: str, transform=None, max_label_length: int = 12, limit: int | None = None):
        self.split = split
        self.transform = transform
        frame = frame[frame["split"].astype(str).str.lower() == split].copy()
        frame["label"] = frame["label"].map(normalize_plate_text)
        frame = frame[frame["label"].str.len().between(1, int(max_label_length))].copy()
        if limit is not None:
            frame = frame.head(int(limit)).copy()
        self.frame = frame.reset_index(drop=True)

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, index: int):
        row = self.frame.iloc[index]
        image_path = Path(row["image_path"])
        image = Image.open(image_path).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        metadata = {
            "split": self.split,
            "source_name": str(row.get("source_name", "")),
            "plate_type": str(row.get("plate_type", "")),
            "label_status": str(row.get("label_status", "")),
            "review_status": str(row.get("review_status", "")),
        }
        return image, row["label"], str(image_path), metadata


def collate_batch(batch):
    images, labels, paths, metadata = zip(*batch)
    return torch.stack(list(images), dim=0), list(labels), list(paths), list(metadata)


def build_official_transform(img_size: Sequence[int] = (32, 128), augment: bool = False, preprocess: bool = False):
    steps = []
    if preprocess:
        steps.append(T.Lambda(preprocess_plate_image))
    if augment:
        steps.extend([
            T.RandomApply([T.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.12, hue=0.03)], p=0.5),
            T.RandomApply([T.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0))], p=0.15),
            T.RandomAffine(degrees=2, translate=(0.02, 0.04), shear=(-2, 2), fill=0),
        ])
    steps.extend([
        T.Resize(tuple(img_size), interpolation=T.InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
    ])
    return T.Compose(steps)


def _make_plate_type_sampler(frame: pd.DataFrame):
    counts = frame["plate_type"].fillna("unknown").value_counts().to_dict()
    weights = frame["plate_type"].fillna("unknown").map(lambda value: 1.0 / counts[value]).to_numpy(dtype="float64")
    return WeightedRandomSampler(torch.as_tensor(weights, dtype=torch.double), num_samples=len(weights), replacement=True)


def make_loaders(cfg: OfficialPARSeqANPRConfig):
    frame = build_plate_dataframe(cfg)
    train_transform = build_official_transform(cfg.img_size, augment=cfg.augment, preprocess=cfg.preprocess)
    eval_transform = build_official_transform(cfg.img_size, augment=False, preprocess=cfg.preprocess)
    train_ds = PlateCropDataset(frame, "train", train_transform, cfg.max_label_length)
    val_ds = PlateCropDataset(frame, "val", eval_transform, cfg.max_label_length)
    test_ds = PlateCropDataset(frame, "test", eval_transform, cfg.max_label_length)
    if len(train_ds) == 0:
        raise ValueError("Train split is empty after filtering")
    loader_kwargs = dict(
        batch_size=cfg.batch_size,
        num_workers=cfg.num_workers,
        collate_fn=collate_batch,
        pin_memory=torch.cuda.is_available(),
    )
    sampler = _make_plate_type_sampler(train_ds.frame) if cfg.balance_plate_types else None
    return (
        DataLoader(train_ds, shuffle=sampler is None, sampler=sampler, **loader_kwargs),
        DataLoader(val_ds, shuffle=False, **loader_kwargs),
        DataLoader(test_ds, shuffle=False, **loader_kwargs),
        {"train": train_ds, "val": val_ds, "test": test_ds},
    )


class Tokenizer:
    BOS = "[B]"
    EOS = "[E]"
    PAD = "[P]"

    def __init__(self, charset: str) -> None:
        self._itos = (self.EOS,) + tuple(charset) + (self.BOS, self.PAD)
        self._stoi = {s: i for i, s in enumerate(self._itos)}
        self.eos_id = self._stoi[self.EOS]
        self.bos_id = self._stoi[self.BOS]
        self.pad_id = self._stoi[self.PAD]

    def __len__(self):
        return len(self._itos)

    def _tok2ids(self, tokens: str) -> list[int]:
        return [self._stoi[s] for s in tokens]

    def _ids2tok(self, token_ids: list[int], join: bool = True):
        tokens = [self._itos[i] for i in token_ids]
        return "".join(tokens) if join else tokens

    def encode(self, labels: list[str], device: Optional[torch.device] = None) -> Tensor:
        batch = [
            torch.as_tensor([self.bos_id] + self._tok2ids(y) + [self.eos_id], dtype=torch.long, device=device)
            for y in labels
        ]
        return pad_sequence(batch, batch_first=True, padding_value=self.pad_id)

    def _filter(self, probs: Tensor, ids: Tensor) -> tuple[Tensor, list[int]]:
        ids = ids.tolist()
        try:
            eos_idx = ids.index(self.eos_id)
        except ValueError:
            eos_idx = len(ids)
        ids = ids[:eos_idx]
        probs = probs[: eos_idx + 1]
        return probs, ids

    def decode(self, token_dists: Tensor, raw: bool = False) -> tuple[list[str], list[Tensor]]:
        batch_tokens = []
        batch_probs = []
        for dist in token_dists:
            probs, ids = dist.max(-1)
            if not raw:
                probs, ids = self._filter(probs, ids)
            batch_tokens.append(self._ids2tok(ids, not raw))
            batch_probs.append(probs)
        return batch_tokens, batch_probs


def init_weights(module: nn.Module, name: str = "", exclude: Sequence[str] = ()):
    if any(map(name.startswith, exclude)):
        return
    if isinstance(module, nn.Linear):
        nn.init.trunc_normal_(module.weight, std=0.02)
        if module.bias is not None:
            nn.init.zeros_(module.bias)
    elif isinstance(module, nn.Embedding):
        nn.init.trunc_normal_(module.weight, std=0.02)
        if module.padding_idx is not None:
            module.weight.data[module.padding_idx].zero_()
    elif isinstance(module, nn.Conv2d):
        nn.init.kaiming_normal_(module.weight, mode="fan_out", nonlinearity="relu")
        if module.bias is not None:
            nn.init.zeros_(module.bias)
    elif isinstance(module, (nn.LayerNorm, nn.BatchNorm2d, nn.GroupNorm)):
        nn.init.ones_(module.weight)
        nn.init.zeros_(module.bias)


def apply_named(module: nn.Module, fn, prefix: str = "") -> None:
    for child_name, child in module.named_children():
        name = f"{prefix}.{child_name}" if prefix else child_name
        apply_named(child, fn, name)
    fn(module, prefix)


class DecoderLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward=2048, dropout=0.1, activation="gelu", layer_norm_eps=1e-5):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.cross_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = nn.LayerNorm(d_model, eps=layer_norm_eps)
        self.norm2 = nn.LayerNorm(d_model, eps=layer_norm_eps)
        self.norm_q = nn.LayerNorm(d_model, eps=layer_norm_eps)
        self.norm_c = nn.LayerNorm(d_model, eps=layer_norm_eps)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)
        self.activation = transformer._get_activation_fn(activation)

    def __setstate__(self, state):
        if "activation" not in state:
            state["activation"] = F.gelu
        super().__setstate__(state)

    def forward_stream(self, tgt, tgt_norm, tgt_kv, memory, tgt_mask, tgt_key_padding_mask):
        tgt2, _ = self.self_attn(tgt_norm, tgt_kv, tgt_kv, attn_mask=tgt_mask, key_padding_mask=tgt_key_padding_mask)
        tgt = tgt + self.dropout1(tgt2)
        tgt2, _ = self.cross_attn(self.norm1(tgt), memory, memory)
        tgt = tgt + self.dropout2(tgt2)
        tgt2 = self.linear2(self.dropout(self.activation(self.linear1(self.norm2(tgt)))))
        tgt = tgt + self.dropout3(tgt2)
        return tgt

    def forward(self, query, content, memory, query_mask=None, content_mask=None, content_key_padding_mask=None, update_content=True):
        query_norm = self.norm_q(query)
        content_norm = self.norm_c(content)
        query = self.forward_stream(query, query_norm, content_norm, memory, query_mask, content_key_padding_mask)
        if update_content:
            content = self.forward_stream(content, content_norm, content_norm, memory, content_mask, content_key_padding_mask)
        return query, content


class Decoder(nn.Module):
    __constants__ = ["norm"]

    def __init__(self, decoder_layer, num_layers, norm):
        super().__init__()
        self.layers = transformer._get_clones(decoder_layer, num_layers)
        self.num_layers = num_layers
        self.norm = norm

    def forward(self, query, content, memory, query_mask=None, content_mask=None, content_key_padding_mask=None):
        for i, mod in enumerate(self.layers):
            last = i == len(self.layers) - 1
            query, content = mod(query, content, memory, query_mask, content_mask, content_key_padding_mask, update_content=not last)
        return self.norm(query)


class Encoder(VisionTransformer):
    def __init__(
        self,
        img_size=224,
        patch_size=16,
        in_chans=3,
        embed_dim=768,
        depth=12,
        num_heads=12,
        mlp_ratio=4.0,
        qkv_bias=True,
        drop_rate=0.0,
        attn_drop_rate=0.0,
        drop_path_rate=0.0,
        embed_layer=PatchEmbed,
    ):
        super().__init__(
            img_size,
            patch_size,
            in_chans,
            embed_dim=embed_dim,
            depth=depth,
            num_heads=num_heads,
            mlp_ratio=mlp_ratio,
            qkv_bias=qkv_bias,
            drop_rate=drop_rate,
            attn_drop_rate=attn_drop_rate,
            drop_path_rate=drop_path_rate,
            embed_layer=embed_layer,
            num_classes=0,
            global_pool="",
            class_token=False,
        )

    def forward(self, x):
        return self.forward_features(x)


class TokenEmbedding(nn.Module):
    def __init__(self, charset_size: int, embed_dim: int):
        super().__init__()
        self.embedding = nn.Embedding(charset_size, embed_dim)
        self.embed_dim = embed_dim

    def forward(self, tokens: torch.Tensor):
        return math.sqrt(self.embed_dim) * self.embedding(tokens)


class OfficialPARSeqCore(nn.Module):
    def __init__(
        self,
        num_tokens: int,
        max_label_length: int,
        img_size: Sequence[int],
        patch_size: Sequence[int],
        embed_dim: int,
        enc_num_heads: int,
        enc_mlp_ratio: int,
        enc_depth: int,
        dec_num_heads: int,
        dec_mlp_ratio: int,
        dec_depth: int,
        decode_ar: bool,
        refine_iters: int,
        dropout: float,
    ) -> None:
        super().__init__()
        self.max_label_length = max_label_length
        self.decode_ar = decode_ar
        self.refine_iters = refine_iters
        self.encoder = Encoder(img_size, patch_size, embed_dim=embed_dim, depth=enc_depth, num_heads=enc_num_heads, mlp_ratio=enc_mlp_ratio)
        decoder_layer = DecoderLayer(embed_dim, dec_num_heads, embed_dim * dec_mlp_ratio, dropout)
        self.decoder = Decoder(decoder_layer, num_layers=dec_depth, norm=nn.LayerNorm(embed_dim))
        self.head = nn.Linear(embed_dim, num_tokens - 2)
        self.text_embed = TokenEmbedding(num_tokens, embed_dim)
        self.pos_queries = nn.Parameter(torch.Tensor(1, max_label_length + 1, embed_dim))
        self.dropout = nn.Dropout(p=dropout)
        apply_named(self, partial(init_weights, exclude=["encoder"]))
        nn.init.trunc_normal_(self.pos_queries, std=0.02)

    @property
    def _device(self) -> torch.device:
        return next(self.head.parameters(recurse=False)).device

    def encode(self, img: torch.Tensor):
        return self.encoder(img)

    def decode(self, tgt, memory, tgt_mask=None, tgt_padding_mask=None, tgt_query=None, tgt_query_mask=None):
        n, length = tgt.shape
        null_ctx = self.text_embed(tgt[:, :1])
        tgt_emb = self.pos_queries[:, : length - 1] + self.text_embed(tgt[:, 1:])
        tgt_emb = self.dropout(torch.cat([null_ctx, tgt_emb], dim=1))
        if tgt_query is None:
            tgt_query = self.pos_queries[:, :length].expand(n, -1, -1)
        tgt_query = self.dropout(tgt_query)
        return self.decoder(tgt_query, tgt_emb, memory, tgt_query_mask, tgt_mask, tgt_padding_mask)

    def forward(self, tokenizer: Tokenizer, images: Tensor, max_length: Optional[int] = None) -> Tensor:
        testing = max_length is None
        max_length = self.max_label_length if max_length is None else min(max_length, self.max_label_length)
        bs = images.shape[0]
        num_steps = max_length + 1
        memory = self.encode(images)
        pos_queries = self.pos_queries[:, :num_steps].expand(bs, -1, -1)
        tgt_mask = query_mask = torch.triu(torch.ones((num_steps, num_steps), dtype=torch.bool, device=self._device), 1)

        if self.decode_ar:
            tgt_in = torch.full((bs, num_steps), tokenizer.pad_id, dtype=torch.long, device=self._device)
            tgt_in[:, 0] = tokenizer.bos_id
            logits = []
            for i in range(num_steps):
                j = i + 1
                tgt_out = self.decode(
                    tgt_in[:, :j],
                    memory,
                    tgt_mask[:j, :j],
                    tgt_query=pos_queries[:, i:j],
                    tgt_query_mask=query_mask[i:j, :j],
                )
                p_i = self.head(tgt_out)
                logits.append(p_i)
                if j < num_steps:
                    tgt_in[:, j] = p_i.squeeze().argmax(-1)
                    if testing and (tgt_in == tokenizer.eos_id).any(dim=-1).all():
                        break
            logits = torch.cat(logits, dim=1)
        else:
            tgt_in = torch.full((bs, 1), tokenizer.bos_id, dtype=torch.long, device=self._device)
            tgt_out = self.decode(tgt_in, memory, tgt_query=pos_queries)
            logits = self.head(tgt_out)

        if self.refine_iters:
            query_mask[torch.triu(torch.ones(num_steps, num_steps, dtype=torch.bool, device=self._device), 2)] = 0
            bos = torch.full((bs, 1), tokenizer.bos_id, dtype=torch.long, device=self._device)
            for _ in range(self.refine_iters):
                tgt_in = torch.cat([bos, logits[:, :-1].argmax(-1)], dim=1)
                tgt_padding_mask = (tgt_in == tokenizer.eos_id).int().cumsum(-1) > 0
                tgt_out = self.decode(tgt_in, memory, tgt_mask, tgt_padding_mask, pos_queries, query_mask[:, : tgt_in.shape[1]])
                logits = self.head(tgt_out)
        return logits


class OfficialPARSeqSystem(nn.Module):
    def __init__(
        self,
        charset_train: str = PARSEQ_94_FULL_CHARSET,
        charset_test: str = ANPR_CHARSET,
        max_label_length: int = 25,
        batch_size: int = 384,
        lr: float = 7e-4,
        warmup_pct: float = 0.075,
        weight_decay: float = 0.0,
        img_size: Sequence[int] = (32, 128),
        patch_size: Sequence[int] = (4, 8),
        embed_dim: int = 384,
        enc_num_heads: int = 6,
        enc_mlp_ratio: int = 4,
        enc_depth: int = 12,
        dec_num_heads: int = 12,
        dec_mlp_ratio: int = 4,
        dec_depth: int = 1,
        perm_num: int = 6,
        perm_forward: bool = True,
        perm_mirrored: bool = True,
        decode_ar: bool = True,
        refine_iters: int = 1,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()
        self.hparams = SimpleNamespace(
            charset_train=charset_train,
            charset_test=charset_test,
            max_label_length=max_label_length,
            batch_size=batch_size,
            lr=lr,
            warmup_pct=warmup_pct,
            weight_decay=weight_decay,
            img_size=tuple(img_size),
            patch_size=tuple(patch_size),
            embed_dim=embed_dim,
            enc_num_heads=enc_num_heads,
            enc_mlp_ratio=enc_mlp_ratio,
            enc_depth=enc_depth,
            dec_num_heads=dec_num_heads,
            dec_mlp_ratio=dec_mlp_ratio,
            dec_depth=dec_depth,
            perm_num=perm_num,
            perm_forward=perm_forward,
            perm_mirrored=perm_mirrored,
            decode_ar=decode_ar,
            refine_iters=refine_iters,
            dropout=dropout,
        )
        self.tokenizer = Tokenizer(charset_train)
        self.bos_id = self.tokenizer.bos_id
        self.eos_id = self.tokenizer.eos_id
        self.pad_id = self.tokenizer.pad_id
        self.model = OfficialPARSeqCore(
            len(self.tokenizer),
            max_label_length,
            img_size,
            patch_size,
            embed_dim,
            enc_num_heads,
            enc_mlp_ratio,
            enc_depth,
            dec_num_heads,
            dec_mlp_ratio,
            dec_depth,
            decode_ar,
            refine_iters,
            dropout,
        )
        self.rng = np.random.default_rng()
        self.max_gen_perms = perm_num // 2 if perm_mirrored else perm_num
        self.perm_forward = perm_forward
        self.perm_mirrored = perm_mirrored
        self._manual_device = None

    @property
    def _device(self) -> torch.device:
        return self._manual_device or next(self.parameters()).device

    @_device.setter
    def _device(self, value) -> None:
        self._manual_device = torch.device(value)

    def forward(self, images: Tensor, max_length: Optional[int] = None) -> Tensor:
        return self.model.forward(self.tokenizer, images, max_length)

    def gen_tgt_perms(self, tgt):
        max_num_chars = tgt.shape[1] - 2
        if max_num_chars == 1:
            return torch.arange(3, device=self._device).unsqueeze(0)
        perms = [torch.arange(max_num_chars, device=self._device)] if self.perm_forward else []
        max_perms = math.factorial(max_num_chars)
        if self.perm_mirrored:
            max_perms //= 2
        num_gen_perms = min(self.max_gen_perms, max_perms)
        if max_num_chars < 5:
            selector = [0, 3, 4, 6, 9, 10, 12, 16, 17, 18, 19, 21] if max_num_chars == 4 and self.perm_mirrored else list(range(max_perms))
            perm_pool = torch.as_tensor(list(permutations(range(max_num_chars), max_num_chars)), device=self._device)[selector]
            if self.perm_forward:
                perm_pool = perm_pool[1:]
            perms = torch.stack(perms)
            if len(perm_pool):
                i = self.rng.choice(len(perm_pool), size=num_gen_perms - len(perms), replace=False)
                perms = torch.cat([perms, perm_pool[i]])
        else:
            perms.extend([torch.randperm(max_num_chars, device=self._device) for _ in range(num_gen_perms - len(perms))])
            perms = torch.stack(perms)
        if self.perm_mirrored:
            comp = perms.flip(-1)
            perms = torch.stack([perms, comp]).transpose(0, 1).reshape(-1, max_num_chars)
        bos_idx = perms.new_zeros((len(perms), 1))
        eos_idx = perms.new_full((len(perms), 1), max_num_chars + 1)
        perms = torch.cat([bos_idx, perms + 1, eos_idx], dim=1)
        if len(perms) > 1:
            perms[1, 1:] = max_num_chars + 1 - torch.arange(max_num_chars + 1, device=self._device)
        return perms

    def generate_attn_masks(self, perm):
        sz = perm.shape[0]
        mask = torch.zeros((sz, sz), dtype=torch.bool, device=self._device)
        for i in range(sz):
            query_idx = perm[i]
            masked_keys = perm[i + 1 :]
            mask[query_idx, masked_keys] = True
        content_mask = mask[:-1, :-1].clone()
        mask[torch.eye(sz, dtype=torch.bool, device=self._device)] = True
        query_mask = mask[1:, :-1]
        return content_mask, query_mask


def create_official_parseq_model(cfg: OfficialPARSeqANPRConfig, device: str | torch.device = "cuda"):
    if cfg.experiment != "parseq":
        raise ValueError("This self-contained notebook currently embeds the official `parseq` architecture only.")
    model = OfficialPARSeqSystem(charset_test=cfg.charset_test, decode_ar=cfg.decode_ar, refine_iters=cfg.refine_iters)
    if cfg.pretrained:
        state = torch.hub.load_state_dict_from_url(PRETRAINED_PARSEQ_URL, map_location="cpu", check_hash=True)
        model.model.load_state_dict(state)
    device = torch.device(device)
    model.to(device)
    model._device = device
    return model


def set_decode_mode(model, refine_iters: int, decode_ar: Optional[bool] = None) -> None:
    model.model.refine_iters = int(refine_iters)
    if decode_ar is not None:
        model.model.decode_ar = bool(decode_ar)


def edit_distance(left: str, right: str) -> int:
    left = normalize_plate_text(left)
    right = normalize_plate_text(right)
    if left == right:
        return 0
    previous = list(range(len(right) + 1))
    for i, lc in enumerate(left, start=1):
        current = [i]
        for j, rc in enumerate(right, start=1):
            current.append(min(previous[j] + 1, current[j - 1] + 1, previous[j - 1] + (lc != rc)))
        previous = current
    return previous[-1]


@torch.no_grad()
def greedy_decode(model, images: torch.Tensor, max_length: Optional[int] = None):
    logits = model(images, max_length=max_length)
    probs = logits.softmax(-1)
    preds, token_probs = model.tokenizer.decode(probs)
    preds = [normalize_plate_text(pred) for pred in preds]
    confidences = torch.tensor([float(prob.prod().item()) for prob in token_probs], device=images.device)
    return preds, confidences


@torch.no_grad()
def evaluate(model, loader: DataLoader, device: str | torch.device = "cuda", split_name: str = "val", max_length: int | None = None):
    model.eval()
    device = torch.device(device)
    rows = []
    exact = 0
    edits = 0
    chars = 0
    total = 0
    for images, labels, paths, metadata in tqdm(loader, desc=f"eval {split_name}", leave=False):
        images = images.to(device, non_blocking=True)
        preds, confs = greedy_decode(model, images, max_length=max_length)
        for path, pred, target, conf, meta in zip(paths, preds, labels, confs.detach().cpu().tolist(), metadata):
            target = normalize_plate_text(target)
            dist = edit_distance(pred, target)
            ok = pred == target
            exact += int(ok)
            edits += dist
            chars += max(len(target), 1)
            total += 1
            rows.append({
                "image_path": path,
                "target": target,
                "prediction": pred,
                "exact": ok,
                "edit_distance": dist,
                "confidence": conf,
                **meta,
            })
    metrics = {
        "split": split_name,
        "samples": total,
        "exact_acc": exact / max(total, 1),
        "cer": edits / max(chars, 1),
        "char_acc": 1.0 - edits / max(chars, 1),
        "refine_iters": int(model.model.refine_iters),
        "decode_ar": bool(model.model.decode_ar),
    }
    return metrics, pd.DataFrame(rows)


def summarize_predictions_by_group(predictions: pd.DataFrame, group_cols: Sequence[str] = ("plate_type", "source_name")) -> pd.DataFrame:
    if predictions.empty:
        return pd.DataFrame(columns=[*group_cols, "samples", "exact_acc", "cer", "char_acc"])
    rows = []
    for keys, group in predictions.groupby(list(group_cols), dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)
        edits = group["edit_distance"].sum()
        chars = group["target"].map(lambda value: max(len(str(value)), 1)).sum()
        rows.append({
            **dict(zip(group_cols, keys)),
            "samples": int(len(group)),
            "exact_acc": float(group["exact"].mean()),
            "cer": float(edits / max(chars, 1)),
            "char_acc": float(1.0 - edits / max(chars, 1)),
        })
    return pd.DataFrame(rows).sort_values(list(group_cols)).reset_index(drop=True)


def parseq_plm_loss(system, images: torch.Tensor, labels: list[str]) -> torch.Tensor:
    system._device = images.device
    tgt = system.tokenizer.encode(labels, images.device)
    memory = system.model.encode(images)
    tgt_perms = system.gen_tgt_perms(tgt)
    tgt_in = tgt[:, :-1]
    tgt_out = tgt[:, 1:]
    tgt_padding_mask = (tgt_in == system.pad_id) | (tgt_in == system.eos_id)

    loss = torch.zeros((), dtype=memory.dtype, device=images.device)
    loss_numel = 0
    n = int((tgt_out != system.pad_id).sum().item())
    for i, perm in enumerate(tgt_perms):
        tgt_mask, query_mask = system.generate_attn_masks(perm)
        out = system.model.decode(tgt_in, memory, tgt_mask, tgt_padding_mask, tgt_query_mask=query_mask)
        logits = system.model.head(out).flatten(end_dim=1)
        loss = loss + n * F.cross_entropy(logits, tgt_out.flatten(), ignore_index=system.pad_id)
        loss_numel += n
        if i == 1:
            tgt_out = torch.where(tgt_out == system.eos_id, system.pad_id, tgt_out)
            n = int((tgt_out != system.pad_id).sum().item())
    return loss / max(loss_numel, 1)


def train_one_epoch(model, loader: DataLoader, optimizer, scaler, cfg: OfficialPARSeqANPRConfig, device: str | torch.device, epoch: int):
    model.train()
    device = torch.device(device)
    totals = {"loss": 0.0, "samples": 0}
    for images, labels, _paths, _metadata in tqdm(loader, desc=f"train epoch {epoch}", leave=False):
        images = images.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=bool(cfg.amp and device.type == "cuda")):
            loss = parseq_plm_loss(model, images, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        scaler.step(optimizer)
        scaler.update()
        batch_size = images.shape[0]
        totals["loss"] += float(loss.detach().item()) * batch_size
        totals["samples"] += batch_size
    return {"train_loss": totals["loss"] / max(totals["samples"], 1), "train_samples": totals["samples"]}


def save_checkpoint(path: str | Path, model, cfg: OfficialPARSeqANPRConfig, epoch: int, metrics: dict) -> None:
    payload = {
        "model_state_dict": model.state_dict(),
        "config": asdict(cfg),
        "epoch": int(epoch),
        "metrics": metrics,
        "architecture": "official_parseq_embedded_in_notebook",
    }
    torch.save(payload, path)


def load_checkpoint(path: str | Path, device: str | torch.device = "cuda"):
    checkpoint = torch.load(path, map_location=device)
    cfg = OfficialPARSeqANPRConfig(**checkpoint["config"])
    cfg.pretrained = False
    model = create_official_parseq_model(cfg, device=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model._device = torch.device(device)
    return model, cfg, checkpoint


def evaluate_refinement_sweep(model, loader: DataLoader, refine_iters_list: Iterable[int] = (0, 1, 2, 3), device: str | torch.device = "cuda", split_name: str = "val", max_length: int | None = None):
    original_refine_iters = int(model.model.refine_iters)
    original_decode_ar = bool(model.model.decode_ar)
    rows = []
    predictions = {}
    try:
        for refine_iters in refine_iters_list:
            set_decode_mode(model, refine_iters=int(refine_iters), decode_ar=original_decode_ar)
            metrics, preds = evaluate(model, loader, device=device, split_name=f"{split_name}_refine_{refine_iters}", max_length=max_length)
            rows.append(metrics)
            predictions[int(refine_iters)] = preds
    finally:
        set_decode_mode(model, refine_iters=original_refine_iters, decode_ar=original_decode_ar)
    return pd.DataFrame(rows), predictions


repo d:\NEO\LPR
device cuda


## 1. Configuration

`charset_train` giu dung 94-full cua official pretrained weight de khop shape `text_embed` va `head`. `charset_test` va normalize/evaluate duoc gioi han cho bien so ANPR uppercase. Pipeline ben duoi co the tron nhieu dataset source: prepared OCR CSV, folder `labels.txt` theo mau bien, va collected/reviewed CSV.


In [10]:
multi_type_sources = [
    {
        "name": "vietnam_normal",
        "path": str(REPO_DIR / "ocr_dataset_rescued_bbox_new"),
        "format": "prepared_csv",
        "plate_type": "normal",
    },
    {
        "name": "icpr_color_filtered",
        "path": str(REPO_DIR / "baseline_icpr_2026" / "color_filtered"),
        "format": "color_filtered_labels_txt",
    },
    {
        "name": "quandoi",
        "path": str(WORKSPACE_DIR / "collect" / "update_label" / "labled_quandoi"),
        "format": "collected_review_csv",
        "plate_type": "quandoi",
    },
    {
        "name": "ngoaigiao",
        "path": str(WORKSPACE_DIR / "collect" / "update_label" / "label_ngoaigiao"),
        "format": "collected_review_csv",
        "plate_type": "ngoaigiao",
    },
]

cfg = OfficialPARSeqANPRConfig(
    data_root=str(REPO_DIR / "ocr_dataset_rescued_bbox_new"),
    dataset_sources=multi_type_sources,
    output_dir=str(DEFAULT_OUTPUT_DIR),
    experiment="parseq",
    pretrained=True,
    decode_ar=True,
    refine_iters=1,
    preprocess=True,
    augment=True,
    batch_size=16,
    num_workers=0,
    epochs=30,
    lr=1e-5,
    weight_decay=1e-4,
    grad_clip=20.0,
    seed=42,
    amp=True,
    split_val_ratio=0.10,
    split_test_ratio=0.10,
    exclude_rejected_labels=True,
    balance_plate_types=True,
)

set_seed(cfg.seed)
out_dir = Path(cfg.output_dir)
out_dir.mkdir(parents=True, exist_ok=True)
print(json.dumps(cfg.__dict__, indent=2))


{
  "data_root": "d:\\NEO\\LPR\\ocr_dataset_rescued_bbox_new",
  "dataset_sources": [
    {
      "name": "vietnam_normal",
      "path": "d:\\NEO\\LPR\\ocr_dataset_rescued_bbox_new",
      "format": "prepared_csv",
      "plate_type": "normal"
    },
    {
      "name": "icpr_color_filtered",
      "path": "d:\\NEO\\LPR\\baseline_icpr_2026\\color_filtered",
      "format": "color_filtered_labels_txt"
    },
    {
      "name": "quandoi",
      "path": "d:\\NEO\\collect\\update_label\\labled_quandoi",
      "format": "collected_review_csv",
      "plate_type": "quandoi"
    },
    {
      "name": "ngoaigiao",
      "path": "d:\\NEO\\collect\\update_label\\label_ngoaigiao",
      "format": "collected_review_csv",
      "plate_type": "ngoaigiao"
    }
  ],
  "output_dir": "d:\\NEO\\LPR\\model\\parseq_official_anpr_multitype_refine",
  "experiment": "parseq",
  "pretrained": true,
  "decode_ar": true,
  "refine_iters": 1,
  "charset_test": "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ",
  "img_si

## 2. Dataset And Loaders


In [11]:
train_loader, val_loader, test_loader, datasets = make_loaders(cfg)
print({k: len(v) for k, v in datasets.items()})
all_dataset_rows = pd.concat([dataset.frame for dataset in datasets.values()], ignore_index=True)
dataset_summary = summarize_dataset_frame(all_dataset_rows)
dataset_summary.to_csv(out_dir / "dataset_multitype_summary.csv", index=False)
display(dataset_summary)
display(datasets["train"].frame.head())


loaded  3677 rows from vietnam_normal (prepared_csv)
loaded  3678 rows from icpr_color_filtered (color_filtered_labels_txt)
loaded   343 rows from quandoi (collected_review_csv)
loaded    57 rows from ngoaigiao (collected_review_csv)
{'train': 5844, 'val': 948, 'test': 963}


,split,plate_type,source_name,samples
0,test,blue,icpr_color_filtered,1
1,test,ngoaigiao,ngoaigiao,5
2,test,normal,vietnam_normal,552
3,test,other,icpr_color_filtered,355
4,test,quandoi,quandoi,38
5,test,yellow,icpr_color_filtered,12
6,train,blue,icpr_color_filtered,1
7,train,ngoaigiao,ngoaigiao,49
8,train,normal,vietnam_normal,2574
9,train,other,icpr_color_filtered,2840


,split,image_path,label,source_name,plate_type,label_status,review_status
0,train,d:\NEO\LPR\ocr_dataset_rescued_bbox_new\images...,51G10096,vietnam_normal,normal,prepared,prepared
1,train,d:\NEO\LPR\ocr_dataset_rescued_bbox_new\images...,51G10096,vietnam_normal,normal,prepared,prepared
2,train,d:\NEO\LPR\ocr_dataset_rescued_bbox_new\images...,51A65474,vietnam_normal,normal,prepared,prepared
3,train,d:\NEO\LPR\ocr_dataset_rescued_bbox_new\images...,51G51936,vietnam_normal,normal,prepared,prepared
4,train,d:\NEO\LPR\ocr_dataset_rescued_bbox_new\images...,51G51936,vietnam_normal,normal,prepared,prepared


## 3. Load Embedded Official PARSeq Pretrained Model

Model nay la kien truc official PARSeq da duoc dinh nghia truc tiep trong notebook. Iterative refinement nam trong `model.model.refine_iters` va chay ben trong `forward()`.


In [12]:
model = create_official_parseq_model(cfg, device=DEVICE)
model.train()
print(type(model))
print("img_size", model.hparams.img_size)
print("max_label_length", model.hparams.max_label_length)
print("tokenizer size", len(model.tokenizer))
print("decode_ar", model.model.decode_ar)
print("refine_iters", model.model.refine_iters)


<class '__main__.OfficialPARSeqSystem'>
img_size (32, 128)
max_label_length 25
tokenizer size 97
decode_ar True
refine_iters 1


## 4. Baseline Refinement Sweep Before Fine-Tuning

Cell n?y tr? l?i c?u h?i refinement c? gi?p g? tr?n official pretrained weight tr??c khi fine-tune ANPR hay kh?ng.


In [13]:
baseline_refine_df, baseline_refine_preds = evaluate_refinement_sweep(
    model,
    val_loader,
    refine_iters_list=(0, 1, 2, 3),
    device=DEVICE,
    split_name="val_before_finetune",
    max_length=cfg.max_label_length,
)
baseline_refine_df.to_csv(out_dir / "baseline_refinement_sweep_val.csv", index=False)
display(baseline_refine_df)


eval val_before_finetune_refine_0:   0%|          | 0/60 [00:00<?, ?it/s]

eval val_before_finetune_refine_1:   0%|          | 0/60 [00:00<?, ?it/s]

eval val_before_finetune_refine_2:   0%|          | 0/60 [00:00<?, ?it/s]

eval val_before_finetune_refine_3:   0%|          | 0/60 [00:00<?, ?it/s]

,split,samples,exact_acc,cer,char_acc,refine_iters,decode_ar
0,val_before_finetune_refine_0,948,0.223629,0.396831,0.603169,0,True
1,val_before_finetune_refine_1,948,0.221519,0.397460,0.602540,1,True
2,val_before_finetune_refine_2,948,0.221519,0.397837,0.602163,2,True
3,val_before_finetune_refine_3,948,0.221519,0.397837,0.602163,3,True


## 5. Supervised Fine-Tuning With Official PARSeq PLM Loss

Loss trong notebook copy sat logic official PARSeq: encode image, sinh target permutations, generate attention masks, roi tinh CE tren cac permutation.


In [14]:
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scaler = torch.cuda.amp.GradScaler(enabled=bool(cfg.amp and DEVICE == "cuda"))

history = []
best_val_exact = -1.0
best_path = out_dir / "best_official_parseq_anpr.pt"

for epoch in range(1, cfg.epochs + 1):
    start = time.time()
    train_metrics = train_one_epoch(model, train_loader, optimizer, scaler, cfg, DEVICE, epoch)
    set_decode_mode(model, refine_iters=cfg.refine_iters, decode_ar=cfg.decode_ar)
    val_metrics, _ = evaluate(model, val_loader, device=DEVICE, split_name="val", max_length=cfg.max_label_length)
    row = {
        "epoch": epoch,
        **train_metrics,
        **{f"val_{k}": v for k, v in val_metrics.items()},
        "seconds": time.time() - start,
    }
    history.append(row)
    print(row)
    if val_metrics["exact_acc"] > best_val_exact:
        best_val_exact = val_metrics["exact_acc"]
        save_checkpoint(best_path, model, cfg, epoch, val_metrics)
        print("saved", best_path)

history_df = pd.DataFrame(history)
history_df.to_csv(out_dir / "history.csv", index=False)
display(history_df)


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_36172\2250140460.py:2: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=bool(cfg.amp and DEVICE == "cuda"))


train epoch 1:   0%|          | 0/366 [00:00<?, ?it/s]

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_36172\1594054104.py:1043: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=bool(cfg.amp and device.type == "cuda")):


eval val:   0%|          | 0/60 [00:00<?, ?it/s]

{'epoch': 1, 'train_loss': 0.7298982390224893, 'train_samples': 5844, 'val_split': 'val', 'val_samples': 948, 'val_exact_acc': 0.34282700421940926, 'val_cer': 0.25587828492392806, 'val_char_acc': 0.7441217150760719, 'val_refine_iters': 1, 'val_decode_ar': True, 'seconds': 109.2870671749115}
saved d:\NEO\LPR\model\parseq_official_anpr_multitype_refine\best_official_parseq_anpr.pt


...

{'epoch': 29, 'train_loss': 0.015410365685017385, 'train_samples': 5844, 'val_split': 'val', 'val_samples': 948, 'val_exact_acc': 0.959915611814346, 'val_cer': 0.005532503457814661, 'val_char_acc': 0.9944674965421854, 'val_refine_iters': 1, 'val_decode_ar': True, 'seconds': 99.69817233085632}
saved d:\NEO\LPR\model\parseq_official_anpr_multitype_refine\best_official_parseq_anpr.pt


train epoch 30:   0%|          | 0/366 [00:00<?, ?it/s]

eval val:   0%|          | 0/60 [00:00<?, ?it/s]

{'epoch': 30, 'train_loss': 0.014454855993571759, 'train_samples': 5844, 'val_split': 'val', 'val_samples': 948, 'val_exact_acc': 0.9556962025316456, 'val_cer': 0.005783980887715328, 'val_char_acc': 0.9942160191122846, 'val_refine_iters': 1, 'val_decode_ar': True, 'seconds': 99.7035505771637}


,epoch,train_loss,train_samples,val_split,val_samples,val_exact_acc,val_cer,val_char_acc,val_refine_iters,val_decode_ar,seconds
0,1,0.729898,5844,val,948,0.342827,0.255878,0.744122,1,True,109.287067
1,2,0.293931,5844,val,948,0.647679,0.083239,0.916761,1,True,112.104721
2,3,0.196096,5844,val,948,0.812236,0.036967,0.963033,1,True,102.307453
3,4,0.152990,5844,val,948,0.856540,0.024771,0.975229,1,True,100.407638
4,5,0.114557,5844,val,948,0.889241,0.018735,0.981265,1,True,100.000455
5,6,0.097047,5844,val,948,0.901899,0.016346,0.983654,1,True,102.759211
6,7,0.085341,5844,val,948,0.916667,0.013580,0.986420,1,True,101.464559
7,8,0.068482,5844,val,948,0.916667,0.013077,0.986923,1,True,100.983568
8,9,0.062562,5844,val,948,0.927215,0.011065,0.988935,1,True,100.602926
9,10,0.053414,5844,val,948,0.925105,0.011191,0.988809,1,True,100.486989


## 6. Refinement Sweep After Fine-Tuning

Load checkpoint t?t nh?t r?i sweep l?i `refine_iters = 0/1/2/3` tr?n validation v? test.


In [15]:
best_model, best_cfg, best_ckpt = load_checkpoint(best_path, device=DEVICE)
print("loaded", best_path, "epoch", best_ckpt.get("epoch"), best_ckpt.get("metrics"))

val_refine_df, val_refine_preds = evaluate_refinement_sweep(
    best_model,
    val_loader,
    refine_iters_list=(0, 1, 2, 3),
    device=DEVICE,
    split_name="val_after_finetune",
    max_length=cfg.max_label_length,
)
test_refine_df, test_refine_preds = evaluate_refinement_sweep(
    best_model,
    test_loader,
    refine_iters_list=(0, 1, 2, 3),
    device=DEVICE,
    split_name="test_after_finetune",
    max_length=cfg.max_label_length,
)

val_refine_df.to_csv(out_dir / "refinement_sweep_val.csv", index=False)
test_refine_df.to_csv(out_dir / "refinement_sweep_test.csv", index=False)

best_refine_iters = int(val_refine_df.sort_values("exact_acc", ascending=False).iloc[0]["refine_iters"])
val_by_plate_type = summarize_predictions_by_group(val_refine_preds[best_refine_iters])
test_by_plate_type = summarize_predictions_by_group(test_refine_preds[best_refine_iters])
val_refine_preds[best_refine_iters].to_csv(out_dir / "eval_val_predictions_best_refine.csv", index=False)
test_refine_preds[best_refine_iters].to_csv(out_dir / "eval_test_predictions_best_refine.csv", index=False)
val_by_plate_type.to_csv(out_dir / "eval_val_by_plate_type.csv", index=False)
test_by_plate_type.to_csv(out_dir / "eval_test_by_plate_type.csv", index=False)

print("best_refine_iters", best_refine_iters)
display(val_refine_df)
display(test_refine_df)
display(val_by_plate_type)
display(test_by_plate_type)


loaded d:\NEO\LPR\model\parseq_official_anpr_multitype_refine\best_official_parseq_anpr.pt epoch 29 {'split': 'val', 'samples': 948, 'exact_acc': 0.959915611814346, 'cer': 0.005532503457814661, 'char_acc': 0.9944674965421854, 'refine_iters': 1, 'decode_ar': True}


eval val_after_finetune_refine_0:   0%|          | 0/60 [00:00<?, ?it/s]

eval val_after_finetune_refine_1:   0%|          | 0/60 [00:00<?, ?it/s]

eval val_after_finetune_refine_2:   0%|          | 0/60 [00:00<?, ?it/s]

eval val_after_finetune_refine_3:   0%|          | 0/60 [00:00<?, ?it/s]

...

best_refine_iters 1


,split,samples,exact_acc,cer,char_acc,refine_iters,decode_ar
0,val_after_finetune_refine_0,948,0.958861,0.006035,0.993965,0,True
1,val_after_finetune_refine_1,948,0.959916,0.005533,0.994467,1,True
2,val_after_finetune_refine_2,948,0.959916,0.005658,0.994342,2,True
3,val_after_finetune_refine_3,948,0.959916,0.006035,0.993965,3,True


,split,samples,exact_acc,cer,char_acc,refine_iters,decode_ar
0,test_after_finetune_refine_0,963,0.942887,0.008697,0.991303,0,True
1,test_after_finetune_refine_1,963,0.943925,0.008448,0.991552,1,True
2,test_after_finetune_refine_2,963,0.944964,0.008324,0.991676,2,True
3,test_after_finetune_refine_3,963,0.944964,0.008324,0.991676,3,True


,plate_type,source_name,samples,exact_acc,cer,char_acc
0,blue,icpr_color_filtered,1,1.000000,0.000000,1.000000
1,ngoaigiao,ngoaigiao,3,1.000000,0.000000,1.000000
2,normal,vietnam_normal,551,0.967332,0.004053,0.995947
3,other,icpr_color_filtered,355,0.960563,0.005371,0.994629
4,quandoi,quandoi,26,0.846154,0.025641,0.974359
5,yellow,icpr_color_filtered,12,0.833333,0.052083,0.947917


,plate_type,source_name,samples,exact_acc,cer,char_acc
0,blue,icpr_color_filtered,1,1.000000,0.000000,1.000000
1,ngoaigiao,ngoaigiao,5,1.000000,0.000000,1.000000
2,normal,vietnam_normal,552,0.934783,0.010232,0.989768
3,other,icpr_color_filtered,355,0.971831,0.004021,0.995979
4,quandoi,quandoi,38,0.868421,0.022222,0.977778
5,yellow,icpr_color_filtered,12,0.750000,0.031250,0.968750


## 7. Save Summary


In [16]:
summary = {
    "config": cfg.__dict__,
    "best_checkpoint": str(best_path),
    "best_epoch": best_ckpt.get("epoch"),
    "best_val_metrics": best_ckpt.get("metrics"),
    "dataset_summary": dataset_summary.to_dict(orient="records"),
    "baseline_refinement_sweep_val": baseline_refine_df.to_dict(orient="records"),
    "after_finetune_refinement_sweep_val": val_refine_df.to_dict(orient="records"),
    "after_finetune_refinement_sweep_test": test_refine_df.to_dict(orient="records"),
    "val_by_plate_type": val_by_plate_type.to_dict(orient="records"),
    "test_by_plate_type": test_by_plate_type.to_dict(orient="records"),
}
(out_dir / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
print(out_dir / "summary.json")


d:\NEO\LPR\model\parseq_official_anpr_multitype_refine\summary.json
